In [ ]:
from __future__ import annotations

import asyncio
import os
from pathlib import Path

import gradio as gr
from dotenv import load_dotenv
from openai import AsyncOpenAI

from betting_agents import run_betting_pipeline
from betting_trace import TraceLineBuffer, live_trace_session


In [ ]:
load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
MODEL_ID = os.getenv("OPENROUTER_BETTING_MODEL", "openai/gpt-4o-mini")

openrouter_client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

MCP_DIR = Path.cwd().resolve()
BETTING_SERVER = MCP_DIR / "betting_server.py"

betting_mcp_params = {"command": "uv", "args": ["run", str(BETTING_SERVER)]}

In [ ]:
async def stream_betting_run():
    buf = TraceLineBuffer()

    out: dict[str, str] = {}
    err: dict[str, BaseException] = {}

    async def run():
        try:
            with live_trace_session(buf):
                result = await run_betting_pipeline(
                    openrouter_client=openrouter_client,
                    model_id=MODEL_ID,
                    mcp_params=betting_mcp_params,
                )
            out["final"] = result.final_output or "(no output)"
        except BaseException as e:
            err["e"] = e

    task = asyncio.create_task(run())
    while not task.done():
        await asyncio.sleep(0.08)
        yield buf.snapshot_markdown() + "\n\n---\n\n_Working on your bet…_"

    if err:
        yield buf.snapshot_markdown() + f"\n\n**Something went wrong:** {err['e']!s}"
        return

    yield buf.snapshot_markdown() + "\n\n---\n\n## Summary\n\n" + out["final"]

In [ ]:
with gr.Blocks(title="Demo betting") as demo:
    out = gr.Markdown()
    go = gr.Button("Place a bet", variant="primary")
    go.click(stream_betting_run, outputs=out)

demo.launch(share=False, inline=True)